In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from torchvision import transforms

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls /content/drive/MyDrive/VideosClips/Fights | wc -l

68


In [4]:
!ls /content/drive/MyDrive/VideosClips/noFights | wc -l

82


In [5]:
INPUT_BASE  = Path("/content/drive/MyDrive/")
OUTPUT_BASE = Path("/content/drive/MyDrive/Tensors")

In [6]:
OUTPUT_BASE = Path("/content/Tensors")

In [7]:
DATASET_DIRS = [
    "RWF_2000",
    "hockey_fights",
    "Peliculas",
    #"standarVideos",
    "REAL_LIFE_VIOLENCE",
    "fight_detection_cctv",
    "VideosClips",            # videos sinteticos originales, alta resolucion
]

In [8]:
N_FRAMES     = 16       # frames por clip (ventana temporal para el LSTM)
IMG_SIZE     = 224      # resolucion de salida
MANIFEST_CSV = OUTPUT_BASE / "aegis_manifest.csv"

In [9]:
MIN_DURATION = 1.5      # segundos minimos para ser util
MAX_DURATION = 60.0     # segundos maximos (elimina outliers)
MAX_FPS      = 120.0    # FPS maximo valido (filtra metadata corrupta)

In [10]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv"}

In [11]:
VIOLENCE_KEYWORDS    = {"fight", "violence", "violent", "fights",
                         "Train_Fight", "Val_Fight"}

In [12]:
NO_VIOLENCE_KEYWORDS = {"nonfight", "nonviolent", "no_violence", "non_violent",
                         "noviolence", "noFights", "no_fight",
                         "Train_NonFight", "Val_NonFight"}

In [13]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [14]:
normalize = transforms.Compose([
    transforms.ToTensor(),                          # (H,W,C) uint8 -> (C,H,W) float [0,1]
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [15]:
def infer_label(video_path: Path) -> str:
    parts = [p.lower() for p in video_path.parts]
    for part in reversed(parts):
        clean = part.replace("-", "").replace("_", "").replace(" ", "")
        for kw in NO_VIOLENCE_KEYWORDS:
            if kw.replace("_", "") in clean:
                return "NoFights"
        for kw in VIOLENCE_KEYWORDS:
            if kw.replace("_", "") in clean:
                return "Fights"
    return "Unknown"

In [16]:
def infer_dataset(video_path: Path) -> str:
    for ds in DATASET_DIRS:
        if ds.lower() in str(video_path).lower():
            return ds
    return "unknown"


In [17]:
def get_video_info(cap: cv2.VideoCapture) -> tuple:
    """Returns(fps, total_frames, duration_sec)  video."""
    fps         = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration    = total_frames / fps if fps > 0 else 0.0
    return fps, total_frames, duration


In [19]:
def get_video_info(cap: cv2.VideoCapture) -> tuple:
    """Return(fps, total_frames, duration_sec) video."""
    fps         = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration    = total_frames / fps if fps > 0 else 0.0
    return fps, total_frames, duration


def extract_frames(video_path: Path, n_frames: int = N_FRAMES) -> torch.Tensor | None:
    """
    Extracts n frames evenly distributed throughout the video.

    For MJPEG (RWF-2000): Use CAP_PROP_POS_MSEC to avoid the incorrect FRAME_COUNT bug in this codec.

    Returns shape(n_frames, 3, IMG_SIZE, IMG_SIZE) tensor

    or None if the video is unprocessable.
    """
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    fps, total_frames, duration = get_video_info(cap)

    # Filtros de exclusion
    if fps <= 0 or fps > MAX_FPS:
        cap.release()
        return None
    if duration < MIN_DURATION or duration > MAX_DURATION:
        cap.release()
        return None
    if total_frames < n_frames:
        cap.release()
        return None

    # Calcular timestamps en milisegundos para cada frame a extraer
    # Distribucion uniforme: de 5% a 95% de la duracion del video
    # (evitar primeros y ultimos frames que pueden ser negro/fade)
    start_ms = duration * 0.05 * 1000
    end_ms   = duration * 0.95 * 1000
    timestamps_ms = np.linspace(start_ms, end_ms, n_frames)

    frames = []
    for ts in timestamps_ms:
        cap.set(cv2.CAP_PROP_POS_MSEC, ts)
        ret, frame = cap.read()

        if not ret or frame is None:
            # Frame no disponible: crear frame negro como placeholder
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            # BGR -> RGB (OpenCV usa BGR, PyTorch usa RGB)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Redimensionar a 224x224
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE),
                               interpolation=cv2.INTER_AREA)

        # Normalizar con ImageNet stats
        frame_tensor = normalize(frame)   # shape: (3, 224, 224)
        frames.append(frame_tensor)

    cap.release()

    if len(frames) != n_frames:
        return None

    # Stack: list of (3,224,224) -> tensor (16, 3, 224, 224)
    return torch.stack(frames)


In [53]:
# MAIN ETL

In [20]:
def run_etl():
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

    # Directorios de salida por clase
    for label in ["Fights", "NoFights"]:
        (OUTPUT_BASE / label).mkdir(exist_ok=True)

    manifest_records = []
    stats = {
        "processed": 0,
        "skipped_label": 0,
        "skipped_duration": 0,
        "skipped_fps": 0,
        "skipped_frames": 0,
        "skipped_read_error": 0,
    }

    # Recolectar todos los videos de todos los datasets
    all_videos = []
    for ds_name in DATASET_DIRS:
        ds_path = INPUT_BASE / ds_name
        if not ds_path.exists():
            print(f"[SKIP] D+NO FOUND DIR {ds_path}")
            continue
        videos = [
            p for p in ds_path.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        ]
        all_videos.extend(videos)
        print(f"[SCAN] {ds_name}: {len(videos)} videos")

    print(f"\n[ETL] Total VIDEOS TO PROCESS: {len(all_videos)}")
    print(f"[ETL] Output: {OUTPUT_BASE}")
    print(f"[ETL] Tensores shape: ({N_FRAMES}, 3, {IMG_SIZE}, {IMG_SIZE})\n")

    for video_path in tqdm(all_videos, desc="ETL Progress"):
        label = infer_label(video_path)
        dataset = infer_dataset(video_path)

        # Excluir videos sin label
        if label == "Unknown":
            stats["skipped_label"] += 1
            continue

        # Extraer frames y convertir a tensor
        tensor = extract_frames(video_path, N_FRAMES)

        if tensor is None:
            # Determinar razon de fallo para estadisticas
            cap = cv2.VideoCapture(str(video_path))
            if cap.isOpened():
                fps, total_frames, duration = get_video_info(cap)
                cap.release()
                if fps > MAX_FPS:
                    stats["skipped_fps"] += 1
                elif duration < MIN_DURATION or duration > MAX_DURATION:
                    stats["skipped_duration"] += 1
                elif total_frames < N_FRAMES:
                    stats["skipped_frames"] += 1
                else:
                    stats["skipped_read_error"] += 1
            else:
                stats["skipped_read_error"] += 1
            continue

        # Nombre del tensor: <dataset>_<stem>_<label>.pt
        # Usar stem del archivo original para trazabilidad
        tensor_name = f"{dataset}_{video_path.stem}.pt"
        tensor_path = OUTPUT_BASE / label / tensor_name

        # Guardar tensor
        torch.save(tensor.half(), tensor_path)

        # Registrar en manifest
        manifest_records.append({
            "tensor_path":  str(tensor_path.relative_to(OUTPUT_BASE)),
            "label":        label,
            "label_int":    1 if label == "Fights" else 0,
            "dataset":      dataset,
            "source_video": str(video_path.relative_to(INPUT_BASE)),
            "n_frames":     N_FRAMES,
            "img_size":     IMG_SIZE,
        })
        stats["processed"] += 1

    # Guardar manifest CSV
    df_manifest = pd.DataFrame(manifest_records)
    df_manifest.to_csv(MANIFEST_CSV, index=False, encoding="utf-8")

    # Reporte final
    print("\n" + "=" * 60)
    print("ETL COMPLETE")
    print("=" * 60)
    print(f"  Tensores     : {stats['processed']:,}")
    print(f"  Saltados (label)   : {stats['skipped_label']:,}")
    print(f"  Saltados (duracion): {stats['skipped_duration']:,}")
    print(f"  Saltados (fps)     : {stats['skipped_fps']:,}")
    print(f"  Saltados (frames)  : {stats['skipped_frames']:,}")
    print(f"  Saltados (error)   : {stats['skipped_read_error']:,}")
    print(f"\n  Manifest CSV: {MANIFEST_CSV}")

    if len(df_manifest) > 0:
        print(f"\n  LABEL DISTRIBUTION:")
        dist = df_manifest["label"].value_counts()
        for label, count in dist.items():
            pct = count / len(df_manifest) * 100
            print(f"    {label:<15}: {count:,} ({pct:.1f}%)")

        print(f"\n  BY DATASET:")
        by_ds = df_manifest.groupby(["dataset", "label"]).size().unstack(fill_value=0)
        print(by_ds.to_string())

    # Estimar tamano en disco
    n_tensors = stats["processed"]
    # Shape (16, 3, 224, 224) float32 = 16*3*224*224*4 bytes = 9,633,792 bytes ~ 9.2 MB por tensor
    bytes_per_tensor = N_FRAMES * 3 * IMG_SIZE * IMG_SIZE * 4
    total_gb = (n_tensors * bytes_per_tensor) / (1024 ** 3)
    print(f"\n  DISK SPACE (estimado):")
    print(f"    {bytes_per_tensor / (1024**2):.1f} MB by tensor ")
    print(f"    {total_gb:.1f} GB on all {n_tensors:,} tensors")
    print(f"\n  PD:  USe torch.save con float16:")
    print(f"    torch.save(tensor.half(), tensor_path)  # 4.6 MB by tensor")



In [ ]:
# DATALOADER

In [21]:
class AegisDataset(torch.utils.data.Dataset):
    """
    Dataset PyTorch , It load videos pre extracted to datasets from manifest scv

    Uso:
        dataset = AegisDataset(manifest_csv, split="train")
        loader  = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

        for tensors, labels in loader:
            # tensors shape: (batch, 16, 3, 224, 224)
            # labels shape : (batch,) con valores 0 o 1
            pass
    """

    def __init__(self, manifest_csv: str,
                 split: str = "all",
                 val_ratio: float = 0.15,
                 seed: int = 42):
        """
        manifest_csv : path al CSV generado por run_etl()
        split        : "train", "val", o "all"
        val_ratio    : validation set
        seed         : reusable seed
        """
        self.base_dir = OUTPUT_BASE
        df = pd.read_csv(manifest_csv)

        if split != "all":
            # Split estratificado por label para mantener balance
            from sklearn.model_selection import train_test_split
            train_df, val_df = train_test_split(
                df,
                test_size=val_ratio,
                stratify=df["label_int"],
                random_state=seed
            )
            self.df = train_df.reset_index(drop=True) if split == "train" \
                      else val_df.reset_index(drop=True)
        else:
            self.df = df.reset_index(drop=True)

        print(f"[AegisDataset] split={split}, samples={len(self.df):,}")
        print(f"  Fights   : {(self.df['label_int']==1).sum():,}")
        print(f"  NoFights : {(self.df['label_int']==0).sum():,}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tensor_path = self.base_dir / row["tensor_path"]

        # Cargar tensor pre-extraido: shape (16, 3, 224, 224)
        tensor = torch.load(tensor_path, weights_only=True)

        label = torch.tensor(row["label_int"], dtype=torch.float32)
        return tensor, label


In [22]:
if __name__ == "__main__":
    print("=" * 60)
    print("AegisSentinel-Net — ETL Pipeline")
    print("=" * 60)
    print(f"Input  : {INPUT_BASE}")
    print(f"Output : {OUTPUT_BASE}")
    print(f"Frames : {N_FRAMES} per video")
    print(f"Size   : {IMG_SIZE}x{IMG_SIZE}")
    print(f"Filters: duration [{MIN_DURATION}s, {MAX_DURATION}s], fps <= {MAX_FPS}")
    print()

    run_etl()

AegisSentinel-Net — ETL Pipeline
Input  : /content/drive/MyDrive
Output : /content/Tensors
Frames : 16 per video
Size   : 224x224
Filters: duration [1.5s, 60.0s], fps <= 120.0

[SCAN] RWF_2000: 984 videos
[SCAN] hockey_fights: 1000 videos
[SCAN] Peliculas: 201 videos
[SCAN] REAL_LIFE_VIOLENCE: 2001 videos
[SCAN] fight_detection_cctv: 300 videos
[SCAN] VideosClips: 152 videos

[ETL] Total VIDEOS TO PROCESS: 4638
[ETL] Output: /content/Tensors
[ETL] Tensores shape: (16, 3, 224, 224)



ETL Progress: 100%|██████████| 4638/4638 [2:10:21<00:00,  1.69s/it]



ETL COMPLETE
  Tensores     : 4,632
  Saltados (label)   : 1
  Saltados (duracion): 5
  Saltados (fps)     : 0
  Saltados (frames)  : 0
  Saltados (error)   : 0

  Manifest CSV: /content/Tensors/aegis_manifest.csv

  LABEL DISTRIBUTION:
    NoFights       : 2,331 (50.3%)
    Fights         : 2,301 (49.7%)

  BY DATASET:
label                 Fights  NoFights
dataset                               
Peliculas                100       101
REAL_LIFE_VIOLENCE       998       999
RWF_2000                 485       499
VideosClips               69        82
fight_detection_cctv     149       150
hockey_fights            500       500

  DISK SPACE (estimado):
    9.2 MB by tensor 
    41.6 GB on all 4,632 tensors

  PD:  USe torch.save con float16:
    torch.save(tensor.half(), tensor_path)  # 4.6 MB by tensor


In [23]:
import shutil

In [24]:
shutil.make_archive(
    "/content/drive/MyDrive/VideoTensors",
    "zip",
    "/content/Tensors"
)

'/content/drive/MyDrive/VideoTensors.zip'

In [26]:
!ls -lah /content/drive/MyDrive/VideoTensors.zip

-rw------- 1 root root 8.6G May 15 17:30 /content/drive/MyDrive/VideoTensors.zip
